**Connect To an LLM and Send Email on Exit**

In [2]:
import google.generativeai as genai
from google.colab import userdata
import smtplib
from email.mime.text import MIMEText

In [3]:
# Configure Gemini API
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # Choose a suitable model (you might need to change this based on the output)
    model = genai.GenerativeModel('gemini-2.5-flash-lite') # Using a potentially available free model
    chat = model.start_chat(history=[])
    print("Gemini model configured and chat started.")
except Exception as e:
    print(f"Error configuring Gemini API: {e}")
    print("Please make sure you have added your GOOGLE_API_KEY to Colab secrets.")

Gemini model configured and chat started.


In [4]:
# Email configuration (will be used later)
EMAIL_ADDRESS = userdata.get('EMAIL')
# IMPORTANT: Use an App Password if you have 2-Factor Authentication enabled
# Replace 'YOUR_APP_PASSWORD_OR_PASSWORD' with your actual password or App Password
EMAIL_PASSWORD = userdata.get('YOUR_APP_PASSWORD_OR_PASSWORD')
SMTP_SERVER = 'smtp.gmail.com'
SMTP_PORT = 587 # or 465 for SSL

In [5]:
# Basic chat loop
print("Agent started. Type 'bye' to end the chat and send the summary.")
chat_history = []

while True:
    user_input = input("You: ")
    if user_input.lower() == 'bye':
        break

    try:
        # Send user message to Gemini and get response
        response = chat.send_message(user_input)
        agent_response = response.text
        print(f"Agent: {agent_response}")

        # Store conversation history
        chat_history.append(f"You: {user_input}")
        chat_history.append(f"Agent: {agent_response}")

    except Exception as e:
        print(f"Error communicating with Gemini: {e}")
        agent_response = "Sorry, I couldn't process that."
        print(f"Agent: {agent_response}")
        chat_history.append(f"You: {user_input}")
        chat_history.append(f"Agent: {agent_response}")

Agent started. Type 'bye' to end the chat and send the summary.
You: who are you ?
Agent: I am a large language model, trained by Google.
You: bye


In [6]:
# Email sending part
print("Chat ended. Sending summary via email...")

try:
    # Create the email content
    email_body = "\n".join(chat_history)
    msg = MIMEText(email_body)
    msg['Subject'] = "Gemini Agent Chat Summary"
    msg['From'] = EMAIL_ADDRESS
    msg['To'] = EMAIL_ADDRESS # Sending the summary to the same address

    # Connect to the SMTP server and send the email
    with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
        server.starttls()  # Secure the connection
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.sendmail(EMAIL_ADDRESS, EMAIL_ADDRESS, msg.as_string())

    print("Chat summary email sent successfully.")

except Exception as e:
    print(f"Error sending email: {e}")

Chat ended. Sending summary via email...
Chat summary email sent successfully.
